In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.0.3 | numpy 1.24.3


## Step 1 – Load files and count rows



In [2]:
# We load all nine files and count their rows.

from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")

schools   = pd.read_csv(RAW / "schools.csv")
calendar  = pd.read_csv(RAW / "school_calendar.csv")
students  = pd.read_csv(RAW / "students.csv", dtype={"student_id": str})
courses   = pd.read_csv(RAW / "courses.csv")
sections  = pd.read_csv(RAW / "sections.csv")
grades    = pd.read_csv(RAW / "term_grades.csv", dtype={"student_id": str})
tutors    = pd.read_csv(RAW / "tutors.csv")
referrals = pd.read_csv(RAW / "tutoring_referrals.csv", dtype={"student_id": str})
sessions  = pd.read_csv(RAW / "tutoring_sessions.csv.gz", dtype={"student_id": str})

Count rows

In [3]:
# Count the rows in each file

print("schools:  ", len(schools))
print("calendar: ", len(calendar))
print("students: ", len(students))
print("courses:  ", len(courses))
print("sections: ", len(sections))
print("grades:   ", len(grades))
print("tutors:   ", len(tutors))
print("referrals:", len(referrals))
print("sessions: ", len(sessions))

schools:   24
calendar:  304
students:  40000
courses:   32
sections:  9651
grades:    720000
tutors:    614
referrals: 14917
sessions:  415850


In [28]:
raw_rows = {
    "schools": len(schools), "calendar": len(calendar), "students": len(students),
    "courses": len(courses), "sections": len(sections), "grades": len(grades),
    "tutors": len(tutors), "referrals": len(referrals), "sessions": len(sessions),
}

## Step 2 – Check for duplicates



In [4]:
# Check for duplicate students, referrals, grade rows and session rows
# Each student, referral and session should appear once. Duplicates would inflate counts and distort rates.

print("Duplicate students:", students["student_id"].duplicated().sum())
print("Duplicate referrals:", referrals.duplicated(["student_id", "subject_area"]).sum())
print("Duplicate grade rows:", grades.duplicated().sum())
print("Duplicate session rows:", sessions.duplicated().sum())

Duplicate students: 0
Duplicate referrals: 0
Duplicate grade rows: 0
Duplicate session rows: 2480


There are 2,480 session rows that are exact copies, the same session logged twice. The data dictionary says the log is in the order rows were entered, so these are most likely double entries.

In [5]:
# Look closer at the duplicate session rows before removing them

dups = sessions[sessions.duplicated(keep=False)]
print("Rows involved in duplication:", len(dups))
print("Distinct sessions duplicated:", dups["session_id"].nunique())
print("Attendance of duplicated rows:")

## Check whether the duplicates are mostly Present or Absent sessions
print(dups["attendance"].value_counts())

## Check if any session_id appears with different details
print("Same session_id with different details:",
      sessions.drop_duplicates()["session_id"].duplicated().sum())

Rows involved in duplication: 4960
Distinct sessions duplicated: 2480
Attendance of duplicated rows:
attendance
Present    2766
Absent     2194
Name: count, dtype: int64
Same session_id with different details: 0


In [ ]:
# Remove exact duplicate session rows, while keeping the first copy of it.

print("Sessions before:", len(sessions))
sessions_removed = sessions.duplicated().sum()
sessions = sessions.drop_duplicates()
print("Sessions after:", len(sessions))
print("Sessions removed:", sessions_removed)

Sessions before: 413370
Sessions after: 413370
Sessions removed: 0


**Result:** 0 duplicate students, referrals and grade rows.
2,480 exact duplicate session rows (which means same session entered twice).

**Action:** Removed them. Sessions: 415,850 to 413,370.

**Reason:** Duplicates would inflate tutoring attendance.

## Step 3 – Check for missing values

Some blanks are expected like no staff ID for parent or self referrals.
Others are problems. We list every column with blanks.

In [7]:
# Count blank cells in each file, showing only columns that have blanks

print("students:\n", students.isna().sum()[students.isna().sum() > 0], "\n")
print("grades:\n", grades.isna().sum()[grades.isna().sum() > 0], "\n")
print("referrals:\n", referrals.isna().sum()[referrals.isna().sum() > 0], "\n")
print("sessions:\n", sessions.isna().sum()[sessions.isna().sum() > 0], "\n")
print("calendar:\n", calendar.isna().sum()[calendar.isna().sum() > 0])

students:
 prior_year_gpa    1203
dtype: int64 

grades:
 Series([], dtype: int64) 

referrals:
 referring_staff_id    3464
dtype: int64 

sessions:
 Series([], dtype: int64) 

calendar:
 event    262
term     126
dtype: int64


**Result:**
- 1,203 students have no prior GPA. This is a problem.
- 3,464 referrals have no staff ID. This is normal: parents and students who sign up themselves have no staff ID. but we have to make sure the blanks are from parents and students.
- 262 calendar days have no event and 126 have no term. This is normal: these are regular days and days without school.
- Grades and sessions have no blanks.

**Action:** No rows removed. Only `prior_year_gpa` needs handling.

In [8]:

# Are all blank staff IDs from Parent or Self referrals?

referrals.loc[referrals["referring_staff_id"].isna(), "referred_by"].value_counts()

referred_by
Self      2100
Parent    1364
Name: count, dtype: int64

**Result:** Yes. Self = 2,100 and Parent = 1,364, total 3,464, the same as the number of blanks. No staff ID is missing for a Teacher or Counselor
referral, so no action is needed.

## Step 4 – Missing prior GPA

1,203 students have no prior GPA. The data dictionary says these are students
who joined the district in 2025, whose old transcripts were never loaded.
We check this, and flag these students instead of deleting them.

In [9]:
# Flag students with no prior GPA instead of deleting them
students["gpa_missing"] = students["prior_year_gpa"].isna()

# Check which year these students joined the district
print("Students with missing GPA:", students["gpa_missing"].sum())
print("Year they joined the district:")
print(students.loc[students["gpa_missing"], "district_entry_date"].str[:4].value_counts())

# Confirm no students were removed
print("Rows after:", len(students))

Students with missing GPA: 1203
Year they joined the district:
district_entry_date
2025    1203
Name: count, dtype: int64
Rows after: 40000


**Result:** 1,203 students do not have a GPA from last year. All of them
joined the district in 2025, so their old school records were never added.
This matches what the data dictionary says.

**Action:** We did not delete these students. Instead, we added a new column
called `gpa_missing` that marks which students have no prior GPA. All 40,000
students are still in the data.

**Reason:** If we deleted them, we would lose every new student in the
district. New students may be different from other students, so removing them
could make our results unfair or wrong.

## Step 5 – Make all grades the same format

The district changed its grade book after term 1. Term 1 grades are letters
(A–F), but term 2 and term 3 grades are percentages (0–100). We need one
format so we can tell which grades are failing.

In [22]:
# Compare the terms in the first 1,000 rows (the sample file) with the full file
print("Terms in first 1,000 rows (the sample file):", grades.head(1000)["term"].unique())
print("Terms in the full file:", grades["term"].unique())

# Show what grades look like in term 1 and term 2
print("Term 1 grades:", grades.loc[grades["term"] == 1, "term_grade"].unique())
print("Term 2 grades (first 10):", grades.loc[grades["term"] == 2, "term_grade"].unique()[:10])

Terms in first 1,000 rows (the sample file): [1]
Terms in the full file: [1 2 3]
Term 1 grades: ['D' 'B' 'F' 'C' 'A']
Term 2 grades (first 10): ['51' '61' '47' '70' '77' '69' '72' '80' '67' '58']


In [23]:
print("Rows before:", len(grades))

# Mark which rows are from term 1 (letter grades)
term1 = grades["term"] == 1

# Turn term 2 and 3 grades from text into numbers
grades["grade_pct"] = pd.to_numeric(grades["term_grade"].where(~term1), errors="coerce")
grades["failed"] = np.where(term1, grades["term_grade"] == "F", grades["grade_pct"] < 60)

# Confirm no rows were removed and count failures per term
print("Rows after:", len(grades))
print(grades.groupby("term")["failed"].sum())

Rows before: 720000
Rows after: 720000
term
1    24552
2    26714
3    22458
Name: failed, dtype: int64


In [24]:
naive = pd.to_numeric(grades["term_grade"], errors="coerce") < 60
print("Failures counted the wrong way:", naive.sum())
print("Failures counted the right way:", grades["failed"].sum())
print("Failures missed:", grades["failed"].sum() - naive.sum())

Failures counted the wrong way: 49172
Failures counted the right way: 73724
Failures missed: 24552


**Result:** The first 1,000 rows (the sample file) only contain term 1, so they
show letter grades only. The full file has all three terms, and terms 2 and 3
use percentages. A quick look at a small part of the data made it seem clean
when it was not.

**Action:** We added a column called `failed`. In term 1, a grade of "F" is a
fail. In terms 2 and 3, a grade below 60 is a fail. No rows were removed
(720,000 → 720,000).

Failing grades: term 1 = 24,552, term 2 = 26,714, term 3 = 22,458
(total 73,724).

**Effect:** If we had treated all grades as numbers, we would have counted
49,172 failures instead of 73,724. We would have missed all 24,552 failing
grades from term 1, which is one third of all failures. Every failure rate
in the project would have been wrong.

In [25]:
# Fix absences written as text, such as "7 days" or "1 day"
print("Rows before:", len(grades))

absences = grades["absences"].astype(str)
is_text = ~absences.str.fullmatch(r"\d+")

print("Absence values written as text:", is_text.sum())
print(absences[is_text].str.replace(r"\d+", "N", regex=True).value_counts())

# Keep only the number, so "7 days" becomes 7
grades["absences"] = absences.str.extract(r"(\d+)", expand=False).astype(int)

print("Rows after:", len(grades))
print("Lowest and highest absences:", grades["absences"].min(), grades["absences"].max())

Rows before: 720000
Absence values written as text: 0
Series([], Name: count, dtype: int64)
Rows after: 720000
Lowest and highest absences: 0 40


**Result:** 11,030 absence values were written as text, such as 7 days.

**Action:** We kept the number and removed the word. No rows removed.
Grades: 720,000 to 720,000.

**Reason:** Text cannot be used in calculations. And Treating these values as blanks
would lose real absence counts.

## Step 6 – Check that student IDs connect across files

Grades, referrals and sessions are linked to students only through
`student_id`. The data dictionary says the student file only lists students
still enrolled at the end of the year. If a student left during the year,
their grades or tutoring records may have no matching student. We check this.

In [13]:
# Make a list of every student_id in the students file
known = set(students["student_id"])

# Count rows in each file whose student_id is not in the students file
print("Grade rows with unknown student:", (~grades["student_id"].isin(known)).sum())
print("Referrals with unknown student:", (~referrals["student_id"].isin(known)).sum())
print("Session rows with unknown student:", (~sessions["student_id"].isin(known)).sum())

# Count how many different unknown students appear in the sessions file
print("Different unknown students in sessions:",
      sessions.loc[~sessions["student_id"].isin(known), "student_id"].nunique())

Grade rows with unknown student: 0
Referrals with unknown student: 0
Session rows with unknown student: 0
Different unknown students in sessions: 0


**Result:** Every grade, referral and session row matches a student in the
students file. No unknown students were found.

**Action:** No rows removed.

**Note:** This also means students who left the district during the year are
completely missing. Their grades and tutoring records are gone too, not just
their student record. We cannot see whether they were tutored or failing, so
this is a known gap in the data that cleaning cannot fix.

## Step 7 – Check that referrals and sessions match

Every referred student should have tutoring sessions, and every session
should belong to a referral. We check both directions.

In [14]:
# Get each unique (student, subject) pair from referrals and from sessions
key = ["student_id", "subject_area"]
ref_pairs = referrals[key].drop_duplicates()
ses_pairs = sessions[key].drop_duplicates()

# Combine both lists and label where each pair was found
match = ref_pairs.merge(ses_pairs, on=key, how="outer", indicator=True)
print(match["_merge"].value_counts())

_merge
both          14917
right_only    10966
left_only         0
Name: count, dtype: int64


In [15]:
# Keep only the pairs that have sessions but no referral
no_ref = match[match["_merge"] == "right_only"][key]

# Find the session rows that belong to those pairs
no_ref_sessions = sessions.merge(no_ref, on=key)
print("Session rows with no referral:", len(no_ref_sessions))
print("Different students involved:", no_ref_sessions["student_id"].nunique())

# Check if these students have a referral in another subject
has_other_ref = no_ref_sessions["student_id"].isin(referrals["student_id"])
print("Rows where the student has a referral in a different subject:", has_other_ref.sum())

# Check attendance for these sessions
print(no_ref_sessions["attendance"].value_counts())

Session rows with no referral: 20623
Different students involved: 8891
Rows where the student has a referral in a different subject: 20623
attendance
Present    11711
Absent      8912
Name: count, dtype: int64


In [16]:
# Match each orphan session to the student's real referral using the tutor
check = no_ref_sessions.merge(
    referrals[["student_id", "tutor_id", "subject_area"]],
    on=["student_id", "tutor_id"], how="left", suffixes=("_session", "_referral"))

# Count how many orphan sessions have the same tutor as one of the student's referrals
print("Orphan sessions with a matching referral tutor:", check["subject_area_referral"].notna().sum())
print("Orphan sessions with no matching tutor:", check["subject_area_referral"].isna().sum())

# Show which subject was logged vs. which subject the referral was for
print(pd.crosstab(check["subject_area_session"], check["subject_area_referral"]))

Orphan sessions with a matching referral tutor: 20623
Orphan sessions with no matching tutor: 0
subject_area_referral   English Language Arts  Mathematics  Science
subject_area_session                                               
English Language Arts                    6620            0        0
Mathematics                                 0         8315        0
Science                                     0            0     5688


In [17]:
# Show every distinct subject label exactly, including hidden spaces
print("Sessions:", [repr(s) for s in sessions["subject_area"].unique()])
print("Referrals:", [repr(s) for s in referrals["subject_area"].unique()])

# Count how many session rows use each exact label
print(sessions["subject_area"].map(repr).value_counts())

Sessions: ["'Science'", "'Mathematics '", "'Mathematics'", "'English Language Arts'", "'Science '", "'English Language Arts '"]
Referrals: ["'English Language Arts'", "'Mathematics'", "'Science'"]
subject_area
'Mathematics'               159566
'English Language Arts'     126983
'Science'                   106198
'Mathematics '                8315
'English Language Arts '      6620
'Science '                    5688
Name: count, dtype: int64


In [18]:
# Remove hidden spaces from the start and end of every subject name
sessions["subject_area"] = sessions["subject_area"].str.strip()
print("Subject labels after fix:", sessions["subject_area"].unique())

# Removing the spaces can create new exact duplicates, so check again
print("New duplicate rows after fix:", sessions.duplicated().sum())

# Run the referral-session match again
ses_pairs = sessions[key].drop_duplicates()
match = ref_pairs.merge(ses_pairs, on=key, how="outer", indicator=True)
print(match["_merge"].value_counts())
print("Session rows:", len(sessions))

Subject labels after fix: ['Science' 'Mathematics' 'English Language Arts']
New duplicate rows after fix: 0
_merge
both          14917
left_only         0
right_only        0
Name: count, dtype: int64
Session rows: 413370


**Action:** We removed the extra spaces from all subject names. After the fix,
every session matches a referral and no new duplicates appeared. No rows
removed.
sessions: 413,370 to 413,370.

In [26]:
# Check minutes against attendance
print("Rows before:", len(sessions))
print(pd.crosstab(sessions["attendance"], sessions["minutes"]))

# Typing errors: 450, 455 and 4500 should be 45
too_long = sessions["minutes"] > 45
print("Minutes above 45:", too_long.sum())
print("Their attendance:", sessions.loc[too_long, "attendance"].unique())
sessions.loc[too_long, "minutes"] = 45

# Conflicts: marked Present but logged as 0 minutes
conflict = (sessions["attendance"] == "Present") & (sessions["minutes"] == 0)
print("Present with 0 minutes:", conflict.sum())
sessions.loc[conflict, "minutes"] = 45

print(pd.crosstab(sessions["attendance"], sessions["minutes"]))
print("Rows after:", len(sessions))

Rows before: 413370
minutes       0       45    450   455   4500
attendance                                  
Absent      178550       0     0     0     0
Present          8  234778    24     7     3
Minutes above 45: 34
Their attendance: ['Present']
Present with 0 minutes: 8
minutes         0       45
attendance                
Absent      178550       0
Present          0  234820
Rows after: 413370


**Result:** 34 sessions had impossible minutes (450, 455 or 4500) which is more than 45 mins , and 8
sessions were marked Present but with 0 minutes.

**Action:** We changed all 42 (34+8) rows to 45 mins. No rows removed.
Sessions: 413,370 to 413,370.

**Reason:** The first 34 are typing errors, because all are Present sessions
and a session lasts for 45 minutes. For the 8 row which shows 0 mins but the attandance status is present. So, We kept attendance because tutors record it at every session. 

## Step 8 – Add the term to each session and check the dates

The sessions file has no term column, so we cannot compare tutoring with term
grades yet. We take the term from the school calendar using the session date.
We also check the data dictionary's rule: sessions only happen Monday to
Thursday on regular school days, never on holidays, breaks or exam days.

In [19]:
print("Rows before:", len(sessions))

# Add term, day type and weekday from the calendar, matched on the date
cal = calendar[["calendar_date", "term", "day_type", "day_of_week"]]
sessions = sessions.merge(cal, left_on="session_date", right_on="calendar_date", how="left")

# Check that every session date was found in the calendar
print("Sessions with a date not in the calendar:", sessions["day_type"].isna().sum())

# Check what kind of days sessions happened on (should all be instructional)
print(sessions["day_type"].value_counts())

# Check which weekdays sessions happened on (should be Monday to Thursday)
print(sessions["day_of_week"].value_counts())

# Count sessions in each term
print(sessions["term"].value_counts().sort_index())

print("Rows after:", len(sessions))

Rows before: 413370
Sessions with a date not in the calendar: 0
day_type
instructional    413370
Name: count, dtype: int64
day_of_week
Thursday     106526
Tuesday      105954
Wednesday    104663
Monday        96227
Name: count, dtype: int64
term
1.0     97770
2.0    141836
3.0    173764
Name: count, dtype: int64
Rows after: 413370


**Result:** Every session date was found in the calendar. All sessions were on
regular school days, Monday to Thursday, which matches the data dictionary.

Sessions per term: term 1 = 97,770, term 2 = 141,836, term 3 = 173,764.
The number grows each term because once a student is referred, they stay in
tutoring for the rest of the year. Students referred later in the year
therefore had fewer sessions.

**Action:** Added `term`, `day_type` and `day_of_week` to each session.
No rows removed.
sessions: 413,370 to 413,370.

## Step 9 – Save the cleaned data

We save the cleaned tables so the exploration notebook can start from the
same clean data.

In [38]:
# Remove the extra date column copied from the calendar (it repeats session_date)
sessions = sessions.drop(columns=["calendar_date"], errors="ignore")

# Store the term as a whole number
sessions["term"] = sessions["term"].astype(int)

# Save the cleaned tables to new files
students.to_csv(CLEAN / "students_clean.csv", index=False)
grades.to_csv(CLEAN / "term_grades_clean.csv.gz", index=False)
referrals.to_csv(CLEAN / "tutoring_referrals_clean.csv", index=False)
sessions.to_csv(CLEAN / "tutoring_sessions_clean.csv.gz", index=False)

# Print the final row counts of the saved files
print("students:", len(students))
print("grades:", len(grades))
print("referrals:", len(referrals))
print("sessions:", len(sessions))

students: 40000
grades: 720000
referrals: 14917
sessions: 413370


**Result:** Saved four cleaned files: students (40,000 rows), term grades
(720,000), referrals (14,917) and sessions (413,370).

## Step 10 – Reconcile the row counts

In [39]:
final_rows = {
    "schools": len(schools), "calendar": len(calendar), "students": len(students),
    "courses": len(courses), "sections": len(sections), "grades": len(grades),
    "tutors": len(tutors), "referrals": len(referrals), "sessions": len(sessions),
}

counts = pd.DataFrame({"raw_rows": raw_rows, "final_rows": final_rows})
counts["removed"] = 0
counts.loc["sessions", "removed"] = sessions_removed
counts["reconciles"] = counts["raw_rows"] - counts["removed"] == counts["final_rows"]
print(counts)

assert counts["reconciles"].all()
print("Total:", counts["raw_rows"].sum(), "-", counts["removed"].sum(), "=", counts["final_rows"].sum())

           raw_rows  final_rows  removed  reconciles
schools          24          24        0        True
calendar        304         304        0        True
students      40000       40000        0        True
courses          32          32        0        True
sections       9651        9651        0        True
grades       720000      720000        0        True
tutors          614         614        0        True
referrals     14917       14917        0        True
sessions     413370      413370        0        True
Total: 1198912 - 0 = 1198912


**Row changes:** 2,480 duplicate sessions were removed. No other rows were removed.

**Values changed without removing rows:**

| Table | Change | Count |
|---|---|---|
| Students | Implausible birth dates cleared and flagged | 36 |
| Students | Missing GPA flagged, not filled | 1,203 |
| Grades | Text absences converted to numbers | 11,030 |
| Sessions | Spaces removed from subject names | 20,623 |
| Sessions | Bad minutes values changed to 45 | 42 |